In [ ]:
## Day 29 — Capstone Project
"""
End-to-end Student Dropout Risk Dashboard combining:
- Random Forest with class_weight balanced
- Pickle model persistence
- Feature Importance (global)
- SHAP Explainability (individual)
- Live Streamlit deployment

Key achievement: Client-ready application with
explainable AI — not just predictions but WHY.
"""

'\nEnd-to-end Student Dropout Risk Dashboard combining:\n- Random Forest with class_weight balanced\n- Pickle model persistence\n- Feature Importance (global)\n- SHAP Explainability (individual)\n- Live Streamlit deployment\n\nKey achievement: Client-ready application with\nexplainable AI — not just predictions but WHY.\n'

In [ ]:
# ── Cell 1: Complete Data Pipeline ────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score)
import pickle
import shap

np.random.seed(42)
n_enrolled, n_dropout = 142, 58

enrolled = pd.DataFrame({
    'attendance_rate': np.random.normal(75, 8, n_enrolled).clip(40, 100),
    'avg_score':       np.random.normal(68, 10, n_enrolled).clip(30, 100),
    'distance_km':     np.random.normal(10, 4, n_enrolled).clip(1, 30),
    'family_income':   np.random.choice(['low','medium','high'], n_enrolled, p=[0.2,0.5,0.3]),
    'parent_education':np.random.choice(['none','primary','secondary','higher'], n_enrolled, p=[0.1,0.2,0.4,0.3]),
    'dropout': 0
})

dropout = pd.DataFrame({
    'attendance_rate': np.random.normal(52, 8, n_dropout).clip(20, 75),
    'avg_score':       np.random.normal(45, 10, n_dropout).clip(20, 70),
    'distance_km':     np.random.normal(22, 5, n_dropout).clip(5, 40),
    'family_income':   np.random.choice(['low','medium','high'], n_dropout, p=[0.6,0.3,0.1]),
    'parent_education':np.random.choice(['none','primary','secondary','higher'], n_dropout, p=[0.4,0.3,0.2,0.1]),
    'dropout': 1
})

df = pd.concat([enrolled, dropout], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

enc = OrdinalEncoder(categories=[['low','medium','high'],
                                  ['none','primary','secondary','higher']])
df[['family_income_encoded','parent_education_encoded']] = enc.fit_transform(
    df[['family_income','parent_education']])

X = df[['attendance_rate', 'avg_score', 'distance_km',
        'family_income_encoded', 'parent_education_encoded']]
y = df['dropout']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

# Save model and encoder
with open('capstone_model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('capstone_encoder.pkl', 'wb') as f:
    pickle.dump(enc, f)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("✅ Full pipeline complete")
print(f"Dataset shape     : {df.shape}")
print(f"Dropout rate      : {y.mean():.1%}")
print(f"ROC-AUC           : {roc_auc_score(y_test, y_prob):.3f}")
print(f"PR-AUC            : {average_precision_score(y_test, y_prob):.3f}")
print(classification_report(y_test, y_pred))

✅ Full pipeline complete
Dataset shape     : (200, 8)
Dropout rate      : 29.0%
ROC-AUC           : 1.000
PR-AUC            : 1.000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        28
           1       1.00      1.00      1.00        12

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



In [ ]:
# ── Cell 2: Write Capstone Streamlit App ──────────────────────────────────────
app_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import shap

# ── Page Config ────────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="Student Dropout Risk Dashboard",
    page_icon="🎓",
    layout="wide"
)

# ── Load Model and Encoder ─────────────────────────────────────────────────────
@st.cache_resource
def load_model():
    with open("capstone_model.pkl", "rb") as f:
        model = pickle.load(f)
    with open("capstone_encoder.pkl", "rb") as f:
        enc = pickle.load(f)
    return model, enc

model, enc = load_model()

feature_names = ["Attendance Rate", "Avg Score", "Distance (km)",
                 "Family Income", "Parent Education"]

# ── Header ─────────────────────────────────────────────────────────────────────
st.title("🎓 Student Dropout Risk Dashboard")
st.markdown("**Powered by Random Forest + SHAP Explainability**")
st.markdown("---")

# ── Sidebar Inputs ─────────────────────────────────────────────────────────────
st.sidebar.header("📋 Student Profile")
attendance = st.sidebar.slider("Attendance Rate (%)", 20, 100, 70)
score = st.sidebar.slider("Average Score", 20, 100, 65)
distance = st.sidebar.slider("Distance from School (km)", 1, 40, 10)
income = st.sidebar.selectbox("Family Income", ["low", "medium", "high"])
education = st.sidebar.selectbox("Parent Education",
                                  ["none", "primary", "secondary", "higher"])

# ── Prediction ─────────────────────────────────────────────────────────────────
input_df = pd.DataFrame({
    "attendance_rate": [attendance],
    "avg_score": [score],
    "distance_km": [distance],
    "family_income": [income],
    "parent_education": [education]
})

input_df[["family_income_encoded", "parent_education_encoded"]] = enc.transform(
    input_df[["family_income", "parent_education"]]
)

X_input = input_df[["attendance_rate", "avg_score", "distance_km",
                     "family_income_encoded", "parent_education_encoded"]]

prediction = model.predict(X_input)[0]
probability = model.predict_proba(X_input)[0]
dropout_prob = probability[1]

# ── Results Row ────────────────────────────────────────────────────────────────
col1, col2, col3 = st.columns(3)

with col1:
    st.metric("Dropout Risk", f"{dropout_prob:.1%}")

with col2:
    if prediction == 1:
        st.error("⚠️ HIGH RISK — Intervention Needed")
    else:
        st.success("✅ LOW RISK — Student On Track")

with col3:
    st.metric("Attendance", f"{attendance}%")

st.markdown("---")

# ── Two Column Layout ──────────────────────────────────────────────────────────
left, right = st.columns(2)

with left:
    st.subheader("📊 Feature Importance")
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]
    fig1, ax1 = plt.subplots(figsize=(6, 4))
    ax1.barh([feature_names[i] for i in indices[::-1]],
             importances[indices[::-1]],
             color="#3498db", edgecolor="black", alpha=0.8)
    ax1.set_xlabel("Importance Score")
    ax1.set_title("Global Feature Importance")
    ax1.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    st.pyplot(fig1)

with right:
    st.subheader("🔍 SHAP Explanation — This Student")
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_input)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    shap_vals = shap_values[0, :, 1]
    colors = ["#e74c3c" if v > 0 else "#2ecc71" for v in shap_vals]
    ax2.barh(feature_names, shap_vals, color=colors, edgecolor="black", alpha=0.8)
    ax2.axvline(x=0, color="black", linewidth=0.8)
    ax2.set_xlabel("SHAP Value (red = increases risk)")
    ax2.set_title("Why This Prediction?")
    ax2.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    st.pyplot(fig2)

st.markdown("---")
st.caption("Built by Imad Ali | DS/ML Freelancer | KPK, Pakistan")
'''

with open("capstone_app.py", "w") as f:
    f.write(app_code)

print("✅ Capstone app written successfully")

✅ Capstone app written successfully


In [ ]:
# ── Cell 3: Launch Capstone App ────────────────────────────────────────────────
!pip install streamlit -q
!pip install pyngrok -q
from pyngrok import ngrok
import subprocess
import time
!ngrok authtoken 3BtGzS0yiKXa50rRk5mtVNvdUHy_3aaXR8XtvchABG5ZeLqSH
# Kill any existing streamlit processes
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

# Launch streamlit
process = subprocess.Popen([
    "streamlit", "run", "capstone_app.py",
    "--server.port", "8501",
    "--server.headless", "true"
])
time.sleep(5)

# Create tunnel
public_url = ngrok.connect(8501)
print(f"✅ Capstone Dashboard is LIVE at:")
print(f"🌐 {public_url}")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
✅ Capstone Dashboard is LIVE at:
🌐 NgrokTunnel: "https://conductible-rally-delfina.ngrok-free.dev" -> "http://localhost:8501"
